# 02 — RQ2: Paired statistical evaluation of predictive improvement
**RQ2:** Do discordance features improve readmission prediction beyond conventional clinical and observed-care variables?

Compare feature sets B and C on exactly the same held-out encounters. The primary classifier family was selected on validation Set B AP and locked before test evaluation. Use average precision (AP, stepwise precision-recall summary) as the primary metric. All other model families and threshold metrics are secondary descriptive comparisons. No test tuning.

In [ ]:
# Shared input setup (run from the project or statistical_analysis folder).
if (!requireNamespace("jsonlite", quietly=TRUE)) stop("Install jsonlite in R first; see README.")
root <- normalizePath(getwd(), winslash="/", mustWork=TRUE)
while (!file.exists(file.path(root,"statistical_analysis/private_exports/manifest.json"))) {
  parent <- dirname(root)
  if (identical(parent,root)) stop("Run 00_export_for_R.ipynb first.")
  root <- parent
}
input <- file.path(root,"statistical_analysis/private_exports")
output <- file.path(root,"statistical_analysis/outputs")
dir.create(output,recursive=TRUE,showWarnings=FALSE)
meta <- jsonlite::fromJSON(file.path(input,"manifest.json"))
read_split <- function(name) {
  d <- read.csv(file.path(input,paste0(name,".csv")), na.strings=c(""), check.names=FALSE,
                stringsAsFactors=FALSE)
  stopifnot(nrow(d)==meta$rows[[name]], all(d[[meta$target]] %in% c(0,1)),
            !anyNA(d$patient_cluster), all(is.finite(as.matrix(d[meta$discordance]))))
  d
}
train <- read_split("train")
test <- read_split("test")
y <- test[[meta$target]]
stopifnot(length(unique(y))==2)
cat("Training encounters:", nrow(train), "| Test encounters:",nrow(test),"\n")

## Recalculate metrics for all 12 experiments
AP is computed with tie groups, matching scikit-learn's non-interpolated average precision. SVM scores are margins, not probabilities. The binary predictions use the already frozen validation thresholds.

In [ ]:
weighted_ap <- function(y,s,w=rep(1,length(y))) {
  keep <- w>0; y <- y[keep]; s <- s[keep]; w <- w[keep]
  if (!length(y) || sum(w*y)==0 || sum(w*(1-y))==0) return(NA_real_)
  idx <- order(s,decreasing=TRUE); y <- y[idx]; s <- s[idx]; w <- w[idx]
  end <- c(which(diff(s)!=0),length(s))
  tp <- cumsum(w*y)[end]; total <- cumsum(w)[end]
  sum(diff(c(0,tp))/sum(w*y) * tp/total)
}
metric <- function(y,s,pred) {
  tp <- sum(y==1 & pred==1); tn <- sum(y==0 & pred==0)
  fp <- sum(y==0 & pred==1); fn <- sum(y==1 & pred==0)
  precision <- if (tp+fp>0) tp/(tp+fp) else 0
  recall <- tp/(tp+fn)
  c(AP=weighted_ap(y,s),precision=precision,recall=recall,
    F1=if (precision+recall>0) 2*precision*recall/(precision+recall) else 0,
    FNR=fn/(tp+fn),FPR=fp/(fp+tn),TP=tp,TN=tn,FP=fp,FN=fn)
}
m <- meta$models
results <- do.call(rbind,lapply(seq_len(nrow(m)),function(i) {
  data.frame(model=m$model[i],feature_set=m$feature_set[i],
             as.list(metric(y,test[[m$score[i]]],test[[m$prediction[i]]])))
}))
print(results); write.csv(results,file.path(output,"rq2_all_metrics.csv"),row.names=FALSE)
py <- read.csv(file.path(root,"models/outputs/test_metrics_all.csv"))
check <- merge(results,py,by=c("model","feature_set"),suffixes=c("_R","_Python"))
stopifnot(nrow(check)==12,max(abs(check$AP_R-check$AP_Python))<1e-8)
cat("All AP values match Python. No-skill AP:",mean(y),"\n")

In [ ]:
ap <- xtabs(AP~feature_set+model,data=results)
plot_ap <- function() {
  old <- par(mar=c(7,4,3,1)); on.exit(par(old))
  barplot(ap,beside=TRUE,las=2,col=c("grey65","steelblue","darkorange"),
          ylab="Average precision",main="Held-out AP by feature set",ylim=c(0,max(ap,mean(y))*1.2))
  abline(h=mean(y),lty=2); legend("topright",legend=rownames(ap),fill=c("grey65","steelblue","darkorange"),bty="n")
}
plot_ap(); png(file.path(output,"rq2_AP_comparison.png"),width=1500,height=900,res=140); plot_ap(); dev.off()
comparison <- data.frame(model=colnames(ap),AP_A=ap["A",],AP_B=ap["B",],AP_C=ap["C",],delta_C_B=ap["C",]-ap["B",],row.names=NULL)
print(comparison); write.csv(comparison,file.path(output,"rq2_feature_comparison.csv"),row.names=FALSE)

## Primary paired patient-cluster bootstrap
Resample patients with replacement, retaining all their encounters and using identical bootstrap weights for both feature sets. Compute AP(C) − AP(B) in each replicate. The 95% percentile interval is conditional on the already trained models and does not include training variability. A positive interval entirely above zero supports improvement for the primary comparison. An interval including zero is inconclusive, not proof of equivalence. This is not an independent replication of the Python bootstrap.

In [ ]:
primary <- meta$primary_family
pick <- function(label) {
  row <- m[m$model==primary & m$feature_set==label,]
  stopifnot(nrow(row)==1); row
}
b <- test[[pick("B")$score]]; cscore <- test[[pick("C")$score]]
cluster <- match(test$patient_cluster,unique(test$patient_cluster)); n <- max(cluster)
set.seed(42); B <- 1000L
boot <- replicate(B,{
  counts <- tabulate(sample.int(n,n,replace=TRUE),nbins=n); w <- counts[cluster]
  weighted_ap(y,cscore,w)-weighted_ap(y,b,w)
})
boot <- boot[is.finite(boot)]
if (length(boot)<.95*B) stop("Too many invalid bootstrap replicates; inspect cohort.")
ci <- quantile(boot,c(.025,.975),names=FALSE)
primary_result <- data.frame(model=primary,AP_B=weighted_ap(y,b),AP_C=weighted_ap(y,cscore),
  delta_AP=weighted_ap(y,cscore)-weighted_ap(y,b),CI_low=ci[1],CI_high=ci[2],replicates=length(boot))
print(primary_result); write.csv(primary_result,file.path(output,"rq2_primary_bootstrap.csv"),row.names=FALSE)
hist(boot,breaks=35,main="Paired patient bootstrap: AP(C) - AP(B)",xlab="AP difference",col="steelblue"); abline(v=0,lty=2)
png(file.path(output,"rq2_bootstrap.png"),width=1200,height=800,res=140)
hist(boot,breaks=35,main="Paired patient bootstrap: AP(C) - AP(B)",xlab="AP difference",col="steelblue"); abline(v=0,lty=2); dev.off()

## Reporting
Report the primary AP difference and confidence interval regardless of direction. Compare recall, F1 and FNR at locked thresholds as secondary measures, not separate claims of statistical superiority. Do not choose a winner based on test scores or adjust thresholds after seeing results. No clinical utility or savings have been demonstrated.